In [17]:
# Import Packages
from humanloop import Humanloop
from dotenv import load_dotenv
import os
from azure.ai.inference import ChatCompletionsClient
from azure.core.credentials import AzureKeyCredential

In [18]:
# Load the environment variables
load_dotenv()

hl_key = os.getenv('HL_API_KEY')
hl = Humanloop(api_key=hl_key)

selected_model = "llama" #"gpt"

if selected_model == "gpt":
    azure_key = AzureKeyCredential(os.getenv('AZURE_API_KEY_GPT'))
    endpoint= os.getenv('AZURE_ENDPOINT_GPT')
    model = os.getenv('GPT_4O_MINI_MODEL')
else:
    azure_key = AzureKeyCredential(os.getenv('AZURE_API_KEY_LLAMA'))
    endpoint= os.getenv('AZURE_ENDPOINT_LLAMA')
    model = os.getenv('LLAMA_3_3_70B_MODEL')



Attempting to instrument while already instrumented


In [19]:
print(model)

Llama-3-3-70B-Instruct-fraher-ai


In [20]:
# Create the client and call defintion (text is the input from the dataset)

template = [
    {"role": "user", "content": "Classify '{{text}}' according to the Abercrombie distinctiveness scale, choose from: Generic, Descriptive, Suggestive, Arbitrary, or Fanciful. This needs to be as legally accurate as possible, so double-check your answer as peoples lives depend on it. Return only the selected value."},    
]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=azure_key,
    model=model
)


def call_azure(**inputs) -> str:
    response = client.complete(
        messages=hl.prompts.populate_template(inputs=inputs, template=template),        
    )

    return str(response.choices[0].message.content).lower()


In [21]:
# Load the dataset from Humanloop
datapoints_pager = hl.datasets.list_datapoints(id="ds_qLHiVr9aM74HanWmcxDVg", version_id="dsv_18sLL872o8jDhzjdTfUrr")
datapoints = [datapoint for datapoint in datapoints_pager]

In [22]:
# Convert the dataset to support "output" in place of "answer" to support
# Humanloop's default evaluation tool schema
datapoint_cls = type(datapoints[0])  # Get the class type dynamically

modified_datapoints = [
    datapoint_cls(
        inputs=datapoint.inputs,
        messages=datapoint.messages,
        target={"output": str(datapoint.target["answer"]).lower()},  # Rename "answer" to "output"
        id=datapoint.id
    )
    for datapoint in datapoints
]


In [23]:
# Show the default evaluators
[print(x) for x in hl.evaluators.list()]

path='Example Evaluators/AI/Semantic similarity' id='ev_M52eeDx7P0CCUUzWg4xOB' directory_id='dir_xXkiWxaU3fSrV4wS3Gxhj' commit_message='Default Evaluator added by Humanloop' spec=LlmEvaluatorRequest(arguments_type='target_required', return_type='number', attributes=None, options=None, number_limits=EvaluatorJudgmentNumberLimit(min=1.0, max=5.0, step=None), number_valence='positive', evaluator_type='llm', prompt=PromptKernelRequest(model='gpt-4o-mini', endpoint='chat', template=[ChatMessage(content='Measure the degree of similarity between the response and the expected output provided.\nGrade the similarity on a scale of 1 to 5, where 1 is very dissimilar and 5 is very similar.\n\n<response>\n{{ log.output }}\n</response>\n\n<expected_output>\n{{ testcase.target }}\n</expected_output>', name=None, tool_call_id=None, role='system', tool_calls=None)], provider='openai', max_tokens=-1, temperature=0.0, top_p=1.0, stop=None, presence_penalty=0.0, frequency_penalty=0.0, other=None, seed=None

[None, None, None, None, None, None, None, None, None, None]

In [24]:
# Runs the evaluation
hl.evaluations.run(
    name="LegalBench",
    file={
        "path": "Legal Test",
        "callable": call_azure,
        "type": "prompt",
        "version": {"model": model, "template": template},
    },
    dataset={
        "path": "LegalBench",
        "datapoints": modified_datapoints,
    },
    evaluators=[
        {"path": "Example Evaluators/Code/Exact match"},         
        {"path": "Example Evaluators/Code/Levenshtein distance"},
        {"path": "Example Evaluators/Code/Latency"},
    ],
)

Evaluating your prompt function corresponding to `Legal Test` on Humanloop 



Navigate to your Evaluation:
https://app.humanloop.com/project/pr_ZqXNJF7l7aurKvSkCcx1X/evaluations/evr_e01nM665EHuEerbpaAVX3/stats

Prompt Version ID: prv_PyTDXPnpb6KU29Ffr4PW8
Run ID: rn_FO7QlR7JQYPJESaOkOtk8

Running 'Legal Test' over the Dataset 'LegalBench' using 4 workers 
[########################################] 95/95 (100.00%) | DONE0ss

⏳ Evaluation Progress
Total Logs: 1963
Total Judgments: 5794



⏳ Evaluation Progress
Total Logs: 1992
Total Judgments: 5832



⏳ Evaluation Progress
Total Logs: 1992
Total Judgments: 5844



⏳ Evaluation Progress
Total Logs: 1992
Total Judgments: 5886



⏳ Evaluation Progress
Total Logs: 1992
Total Judgments: 5886



⏳ Evaluation Progress
Total Logs: 1992
Total Judgments: 5929



⏳ Evaluation Progress
Total Logs: 1992
Total Judgments: 5976



📊 Evaluation Results for Legal Test 
+----------------------------------------------+---------------------+----------------

[EvaluatorCheck(path='Example Evaluators/Code/Exact match', score=0.51, delta=0.18, threshold=None, threshold_check=None, evaluation_id='evr_e01nM665EHuEerbpaAVX3'),
 EvaluatorCheck(path='Example Evaluators/Code/Levenshtein distance', score=4.15, delta=-1.88, threshold=None, threshold_check=None, evaluation_id='evr_e01nM665EHuEerbpaAVX3'),
 EvaluatorCheck(path='Example Evaluators/Code/Latency', score=0.26, delta=-4.46, threshold=None, threshold_check=None, evaluation_id='evr_e01nM665EHuEerbpaAVX3')]